In [2]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from qopt_funcs import *
from network_funcs import *

In [33]:
PARAMS = {
    'p_det': 0.95,          # Detector efficiency
    'alpha': 0.18,          # Fiber loss (dB/km)
    'q_0': 0.01,            # Baseline QBER (Check)
    'nu': 10**9,            # Repetition rate (1 GHz) !! Check
    'R_dark': 100,          # Dark count rate (Hz)
    'delta_det': 100.E-12,  # Time gate duration (s) !! Check
    'eta_det': 0.95,        # Detector efficiency
    'p_pair': 0.05,         # Pair generation probability
    'eta_c': 0.8,           # Source to fiber coupling

}

In [21]:
N=100
beta = 2.6261                 # \beta param of S2 model
mu = 0.0233
A, dist = S2_graph_definite_N(N, beta, mu, D=2, sample_from_file=False, return_coords=False)

In [34]:
def build_prob_matrix(A, dist):
    T_matrix = 10**(-PARAMS['alpha'] * dist / 10) # Transmitivity matrix
    P_ent_matrix = PARAMS['p_pair'] * PARAMS['eta_c']*PARAMS['p_det'] * T_matrix # Probability of entanglement
    return P_ent_matrix * A # Multiply by Adjacency matrix to keep only existing edges

Probs_mtx = build_prob_matrix(A, dist)

In [39]:
def optimal_quantum_repeater_path(Probs_mtx, source, target=None,  P_BSM=0.5): # structured like nx.single_source_dijkstra
    W = np.zeros(np.shape(Probs_mtx))
    none_zero_edges = Probs_mtx>0
    W[none_zero_edges] = -np.log2(Probs_mtx[none_zero_edges]) - np.log2(P_BSM)
    G = nx.from_numpy_array(W)
    weights, paths = nx.single_source_dijkstra(G, source, target, weight='weight')
    #weights = np.exp(weights + np.log2(P_BSM))   # to avoid overcounting the Bell state success probability
    return weights, paths

weights, path = optimal_quantum_repeater_path(Probs_mtx, 0, target=99)
path

[0, 60, 87, 99]

In [42]:
def calculate_expected_sequential_time(path, Probs_mtx, P_BSM=0.5):
    a = path[0] # First Node
    b = path[1] # Second Node
    T=1/Probs_mtx[a, b]
    for i in range(2,len(path)):
        a = path[i-1]
        b = path[i]
        T_i=1/Probs_mtx[a, b]
        T=(T+T_i)/P_BSM
    return T

total_time = calculate_expected_sequential_time(path, Probs_mtx)
print(total_time)

280.27275895340745


## Post processing functions

In [43]:
def entanglement_generation_rate(total_time):
    return PARAMS['nu']/total_time

def secret_key_rate_BBM92(total_time):
    # This is WRONG (it's the formula for BB84 and not BBM92/E91)
    p_darkcount = PARAMS['R_dark']*PARAMS['delta_det']
    p_signal = 1/total_time
    q_tilde = (0.5*p_darkcount + p_signal*PARAMS['q_0'])/(p_signal + p_darkcount)
    R_ent=entanglement_generation_rate(total_time)
    H = DV_keyrate(q_tilde, p_signal, p_darkcount)
    return R_ent*H
print(entanglement_generation_rate(total_time))
secret_key_rate_BBM92(total_time)

3567952.888943588


np.float64(2991290.7527690334)

Small test